# Data preparation: merging the Meta and Panel samples

This notebook begins with the two raw datasets that already satisfy the study's
quality threshold:

- `meta_dataset.xlsx`
- `panel_dataset.xlsx`

At this stage, no substantive variables are recoded, transformed, or constructed.

The objectives are limited to:

1. importing both datasets;
2. adding a variable identifying the recruitment source;
3. checking respondent identifiers and column compatibility;
4. appending the two samples into one dataset;
5. saving the merged raw dataset.

Because the datasets contain different respondents measured with largely the
same questionnaire, they are appended row-wise rather than joined by an ID.

In [1]:
from pathlib import Path

import pandas as pd

In [2]:
DATA_FOLDER = Path(
    r"C:\Users\JonatanPl\OneDrive - JDC\Desktop\Yoni\HighTechProtest - Sampling Wave 2\Python\excel"
)

META_PATH = DATA_FOLDER / "meta_dataset.xlsx"
PANEL_PATH = DATA_FOLDER / "panel_dataset.xlsx"
OUTPUT_PATH = DATA_FOLDER / "merged_raw_dataset.xlsx"
print(META_PATH.exists())
print(PANEL_PATH.exists())

True
True


## 2. Import the two raw datasets

The first worksheet is imported from each Excel file. The imported dataframes
are kept separate until their basic structure has been checked.

In [3]:
meta_raw = pd.read_excel(
    META_PATH,
    sheet_name="Sheet1"
)

panel_raw = pd.read_excel(
    PANEL_PATH,
    sheet_name="Sheet1"
)

print(f"Meta dataset:  {meta_raw.shape[0]:,} rows × {meta_raw.shape[1]:,} columns")
print(f"Panel dataset: {panel_raw.shape[0]:,} rows × {panel_raw.shape[1]:,} columns")

Meta dataset:  406 rows × 108 columns
Panel dataset: 454 rows × 105 columns


## 3. Preserve the imported datasets

Copies are created so that the originally imported dataframes remain unchanged.
All preparation steps are performed on the copies.

In [4]:
meta = meta_raw.copy()
panel = panel_raw.copy()

## 4. Add the recruitment-source identifier

A new variable named `data_source` identifies whether each respondent was
recruited through Meta or through the Panel sample.

This variable is administrative and does not alter any questionnaire response.

In [5]:
meta["data_source"] = "meta"
panel["data_source"] = "panel"

## 5. Check respondent identifiers

Before appending the datasets, verify that `ResponseId` is unique within each
source and that no identifier appears in both datasets.

In [6]:
id_audit = pd.DataFrame({
    "dataset": ["meta", "panel"],
    "rows": [len(meta), len(panel)],
    "nonmissing_ids": [
        meta["ResponseId"].notna().sum(),
        panel["ResponseId"].notna().sum()
    ],
    "unique_ids": [
        meta["ResponseId"].nunique(dropna=True),
        panel["ResponseId"].nunique(dropna=True)
    ],
    "duplicate_ids": [
        meta["ResponseId"].duplicated().sum(),
        panel["ResponseId"].duplicated().sum()
    ]
})

id_audit

,dataset,rows,nonmissing_ids,unique_ids,duplicate_ids
0,meta,406,406,406,0
1,panel,454,454,454,0


In [7]:
overlapping_ids = set(
    meta["ResponseId"].dropna()
).intersection(
    panel["ResponseId"].dropna()
)

print(f"Response IDs appearing in both datasets: {len(overlapping_ids)}")

Response IDs appearing in both datasets: 0


In [8]:
assert meta["ResponseId"].is_unique, "Duplicate ResponseId values in Meta data."
assert panel["ResponseId"].is_unique, "Duplicate ResponseId values in Panel data."
assert len(overlapping_ids) == 0, "Some ResponseId values occur in both datasets."

## 6. Compare the column structures

The two datasets do not contain exactly the same columns. The comparison below
identifies:

- variables shared by both datasets;
- variables found only in the Meta dataset;
- variables found only in the Panel dataset.

No variables are renamed or deleted at this stage.

In [9]:
meta_columns = set(meta.columns)
panel_columns = set(panel.columns)

shared_columns = sorted(meta_columns.intersection(panel_columns))
meta_only_columns = sorted(meta_columns.difference(panel_columns))
panel_only_columns = sorted(panel_columns.difference(meta_columns))

print(f"Shared columns:     {len(shared_columns)}")
print(f"Meta-only columns:  {len(meta_only_columns)}")
print(f"Panel-only columns: {len(panel_only_columns)}")

Shared columns:     106
Meta-only columns:  3
Panel-only columns: 0


In [10]:
print("Columns found only in the Meta dataset:")
display(pd.DataFrame({"meta_only_variable": meta_only_columns}))

print("Columns found only in the Panel dataset:")
display(pd.DataFrame({"panel_only_variable": panel_only_columns}))

Columns found only in the Meta dataset:


,meta_only_variable
0,Ethnicity_5_TEXT
1,FollowUp_Email
2,FollowUp_Interview


Columns found only in the Panel dataset:


,panel_only_variable


## 7. Append the datasets

`pd.concat()` stacks the respondents from the two sources vertically.

The union of all columns is retained. When a variable exists in only one source,
respondents from the other source receive a missing value for that variable.

`ignore_index=True` creates a new sequential dataframe index. It does not change
the original `ResponseId`.

In [11]:
merged_raw = pd.concat(
    [meta, panel],
    axis=0,
    ignore_index=True,
    sort=False
)

print(
    f"Merged dataset: "
    f"{merged_raw.shape[0]:,} rows × "
    f"{merged_raw.shape[1]:,} columns"
)

Merged dataset: 860 rows × 109 columns


## 8. Verify the merged dataset

The merged dataset should contain every respondent from both source files. The
source counts should equal the original dataset sizes.

In [12]:
merged_raw["data_source"].value_counts(dropna=False)

data_source
panel    454
meta     406
Name: count, dtype: int64

In [13]:
assert len(merged_raw) == len(meta) + len(panel)

assert merged_raw["data_source"].value_counts().to_dict() == {
    "panel": 454,
    "meta": 406
}

assert merged_raw["ResponseId"].is_unique

print("All merge validation checks passed.")

All merge validation checks passed.


## 9. Inspect the first observations from each source

This verifies that the source identifier was assigned correctly and that the
rows from both datasets are present.

In [14]:
display(
    merged_raw.loc[
        merged_raw["data_source"] == "meta",
        ["ResponseId", "data_source"]
    ].head()
)

display(
    merged_raw.loc[
        merged_raw["data_source"] == "panel",
        ["ResponseId", "data_source"]
    ].head()
)

,ResponseId,data_source
0,R_4qaEWnP6Qq0GSX0,meta
1,R_4fEdyzKlFmaKWit,meta
2,R_4D2cy5aFkwMjJji,meta
3,R_4R8RiY0j7aBQNMI,meta
4,R_2Jcdjxe51tfIbbb,meta


,ResponseId,data_source
406,R_4Oq5R7cZYbq1LXm,panel
407,R_4iL7CIYDW8JG9kl,panel
408,R_4DILNZVVQZviJKw,panel
409,R_4xnRpnmOZ3URD57,panel
410,R_4kcU9aKgNvFhkrn,panel


## 10. Save the merged raw dataset

The original source files remain unchanged. The appended dataset is saved as a
new Excel file.

No substantive variables have been recoded or constructed.

In [15]:
merged_raw.to_excel(
    OUTPUT_PATH,
    index=False
)

print(f"Saved merged dataset to: {OUTPUT_PATH.resolve()}")

Saved merged dataset to: C:\Users\JonatanPl\OneDrive - JDC\Desktop\Yoni\HighTechProtest - Sampling Wave 2\Python\excel\merged_raw_dataset.xlsx


In [16]:
final_merge_audit = pd.Series({
    "rows": merged_raw.shape[0],
    "columns": merged_raw.shape[1],
    "meta_cases": merged_raw["data_source"].eq("meta").sum(),
    "panel_cases": merged_raw["data_source"].eq("panel").sum(),
    "unique_response_ids": merged_raw["ResponseId"].nunique(),
    "duplicate_response_ids": merged_raw["ResponseId"].duplicated().sum(),
    "missing_response_ids": merged_raw["ResponseId"].isna().sum()
})

final_merge_audit

rows                      860
columns                   109
meta_cases                406
panel_cases               454
unique_response_ids       860
duplicate_response_ids      0
missing_response_ids        0
dtype: int64